# Load and test Point Transformer v3

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
sys.path.insert(0, str(repo_root / ".."))

In [ ]:
from PointTransformerV3.model import *

### Load Point Transformer V3 weights

In [ ]:
point_transformer = PointTransformerV3(in_channels=4, cls_mode=False, enable_flash=True)
# print(point_transformer)

In [ ]:
import copy

def load_best_effort(model, ckpt_path, map_location="cpu"):
    ckpt = torch.load(ckpt_path, map_location=map_location, weights_only=False)
    sd0 = ckpt.get("state_dict", ckpt)

    model_sd = model.state_dict()

    # candidate transforms applied to checkpoint keys
    def transforms():
        def id_(k): return k
        def rm(pfx):
            return lambda k: k[len(pfx):] if k.startswith(pfx) else k

        cands = [
            ("id", id_),
            ("rm_module", rm("module.")),
            ("rm_model", rm("model.")),
            ("rm_net", rm("net.")),
            ("rm_backbone", rm("backbone.")),
            ("rm_module_model", lambda k: rm("model.")(rm("module.")(k))),
            ("rm_module_backbone", lambda k: rm("backbone.")(rm("module.")(k))),
            ("rm_model_backbone", lambda k: rm("backbone.")(rm("model.")(k))),
        ]
        return cands

    best = None
    for name, tf in transforms():
        sd = {tf(k): v for k, v in sd0.items()}
        match = sum(1 for k, v in sd.items() if k in model_sd and v.shape == model_sd[k].shape)
        if best is None or match > best[0]:
            best = (match, name, sd)

    match, name, sd = best
    # load only shape-compatible keys
    loadable = {k: v for k, v in sd.items() if k in model_sd and v.shape == model_sd[k].shape}
    msg = model.load_state_dict(loadable, strict=False)
    model.eval()

    print(f"[load_best_effort] transform={name}  matched={match}/{len(model_sd)}")
    print("missing_keys:", len(msg.missing_keys), "unexpected_keys:", len(msg.unexpected_keys))
    return msg


In [ ]:
checkpoint_path = "../train/weights/point_transforer_v3_nuscenes_semseg.pth"
msg = load_best_effort(point_transformer, checkpoint_path)

In [ ]:
checkpoint_path = "../train/weights/point_transforer_v3_nuscenes_semseg.pth"
ckpt = torch.load(checkpoint_path, map_location="cpu")

# 1) Extract the actual weights dict
if isinstance(ckpt, dict) and "state_dict" in ckpt:
    sd = ckpt["state_dict"]
    print("Loaded state_dict from 'state_dict' key.")
elif isinstance(ckpt, dict) and "model" in ckpt and isinstance(ckpt["model"], dict):
    sd = ckpt["model"]
    print("Loaded state_dict from 'model' key.")
else:
    sd = ckpt  # already a raw state_dict
    print("Loaded raw state_dict.")

# 2) Strip common prefixes (DDP / Lightning / wrappers)
prefixes = ("module.", "model.", "net.", "backbone.", "rm_model_backbbone")
def strip_prefix(k: str) -> str:
    for p in prefixes:
        if k.startswith(p):
            return k[len(p):]
    return k

sd = {strip_prefix(k): v for k, v in sd.items()}
model_sd = point_transformer.state_dict()
matched = [k for k in sd.keys() if k in model_sd and sd[k].shape == model_sd[k].shape]
print(f"Matched keys: {len(matched)} / {len(model_sd)}  ({100*len(matched)/len(model_sd):.2f}%)")

# optional: show a few examples of what matches / doesn't
print("Example matched:", matched[:5])
missing = [k for k in model_sd.keys() if k not in sd or sd.get(k, None) is None or (k in sd and sd[k].shape != model_sd[k].shape)]
print("Example missing:", missing[:5])
# 3) Load
missing, unexpected = point_transformer.load_state_dict(sd, strict=False)
print("missing:", len(missing))
print("unexpected:", len(unexpected))

point_transformer.eval()

In [ ]:
import time
import zarr
import numpy as np

@torch.inference_mode()
def encode_zarr_sequence(
    zarr_path: str | Path,
    model: torch.nn.Module,
    device: str = "cuda",
    grid_size: float = 0.02,
    use_grid_coord: bool = True,
):
    """
    zarr_path: path to an array shaped (T, N, 3) float32/float64
    Returns:
      feats_list: list length T of torch tensors [Ni, C]
      coords_list: list length T of torch tensors [Ni, 3]
      times_list: list length T of inference times in milliseconds
    """
    zarr_path = Path(zarr_path)
    arr = zarr.open(str(zarr_path), mode="r")
    T, N, _ = arr.shape

    model = model.to(device).eval()

    feats_list, coords_list, times_list = [], [], []
    for t in range(20):
        pc = np.asarray(arr[t])
        mask = np.isfinite(pc).all(axis=1) & (np.abs(pc).sum(axis=1) > 0) &  (pc[:, 0] > 0) & (pc[:, 1] < 20) & (pc[:, 1] > -20)
        pc = pc[mask]

        if pc.shape[0] == 0:
            feats_list.append(torch.empty((0, 0), device=device))
            coords_list.append(torch.empty((0, 3), device=device))
            times_list.append(0.0)
            continue

        coord = torch.from_numpy(pc).to(device=device, dtype=torch.float32)
        feat = torch.cat([coord, torch.zeros((coord.shape[0], 1), device=device)], dim=1)

        batch = torch.zeros((coord.shape[0],), device=device, dtype=torch.long)

        data = {
            "coord": coord,
            "batch": batch,
            "feat": feat,
            "grid_size": torch.tensor(grid_size, device=device),
        }

        if use_grid_coord:
            grid_coord = torch.div(
                coord - coord.min(dim=0).values, grid_size, rounding_mode="trunc"
            ).to(torch.int32)
            data["grid_coord"] = grid_coord

        # Benchmark the model inference
        if device == "cuda":
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        
        out_point = model(data)
        
        if device == "cuda":
            torch.cuda.synchronize()
        end_time = time.perf_counter()
        
        elapsed_ms = (end_time - start_time) * 1000
        times_list.append(elapsed_ms)
        
        feats_list.append(out_point.feat)
        coords_list.append(coord)

    return feats_list, coords_list, times_list

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
feats_list, coords_list, times_list = encode_zarr_sequence(
    "/mnt/home/robotique/datasets/isaacsim/datas/scenario2__run000__Graph_A__leafWaypoint_100__wp1-3__20260108_152608_0/points.zarr",
    point_transformer,
    device=device,
    grid_size=0.02,
)

print(f"Total frames: {len(feats_list)}")
print(f"First frame shape: {feats_list[0].shape}")
print(f"\nInference time statistics:")
print(f"  Mean: {np.mean(times_list):.2f} ms")
print(f"  Median: {np.median(times_list):.2f} ms")
print(f"  Min: {np.min(times_list):.2f} ms")
print(f"  Max: {np.max(times_list):.2f} ms")
print(f"  Std: {np.std(times_list):.2f} ms")

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# --- pick one frame ---
t = 10
coords = coords_list[t].detach().cpu().numpy()   # (N, 3)
feats  = feats_list[t].detach().cpu().numpy()   # (N, C)

mask = (coords[:, 0] >= -15) & (coords[:, 0] <= 15) & \
       (coords[:, 1] >= -15) & (coords[:, 1] <= 15)
coords = coords[mask]
feats  = feats[mask]

# Optional: subsample for responsiveness
max_points = 50000
if coords.shape[0] > max_points:
    idx = np.random.choice(coords.shape[0], size=max_points, replace=False)
    coords = coords[idx]
    feats  = feats[idx]

# --- PCA on features ---
feats_std = StandardScaler(with_mean=True, with_std=True).fit_transform(feats)
pca = PCA(n_components=3, random_state=0)
Z = pca.fit_transform(feats_std)  # (N, 3)

# Use PCA embedding as RGB color (normalize each component to [0,255])
Z_min = Z.min(axis=0, keepdims=True)
Z_max = Z.max(axis=0, keepdims=True)
rgb = (Z - Z_min) / (Z_max - Z_min + 1e-12)
rgb255 = (rgb * 255).astype(np.uint8)

# Plotly wants color as strings like "rgb(r,g,b)"
colors = np.array([f"rgb({r},{g},{b})" for r, g, b in rgb255])

fig = go.Figure(
    data=go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode="markers",
        marker=dict(
            size=2,
            color=colors,   # per-point RGB strings
            opacity=0.85,
        ),
        # Optional hover info
        customdata=Z,
        hovertemplate=(
            "x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<br>"
            "PC1=%{customdata[0]:.2f}<br>PC2=%{customdata[1]:.2f}<br>PC3=%{customdata[2]:.2f}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=f"Point cloud (t={t}) colored by PCA(features) | explained var={pca.explained_variance_ratio_.sum():.2%}",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()